# Test 1

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, isnan, to_timestamp, year, month,
    dayofmonth, dayofweek, date_format, to_date,
    sha2, concat_ws
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# Ép Spark chạy dưới quyền root để HDFS cho phép ghi dữ liệu
os.environ["HADOOP_USER_NAME"] = "root"

# ======================================================================
# KHỞI TẠO SPARK SESSION
# ======================================================================
spark = SparkSession.builder \
    .appName("Retail_Lambda_Architecture_Enriched_No_Duplicate") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,mysql:mysql-connector-java:8.0.33"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# ======================================================================
# CẤU HÌNH MYSQL
# ======================================================================
db_url = "jdbc:mysql://mysql:3306/retail_analytics"
db_props = {
    "driver": "com.mysql.cj.jdbc.Driver",
    "user": "retail_user",
    "password": "retail_pass"
}

# ======================================================================
# HÀM GHI DIMENSION KHÔNG TRÙNG
# ======================================================================
def write_new_records_only(df, table_name, key_columns):
    """
    Chỉ ghi những dòng chưa tồn tại trong MySQL.
    Dùng cho các bảng dimension để tránh duplicate qua nhiều micro-batch.
    """

    try:
        # Đọc key đã tồn tại trong MySQL
        existing_df = spark.read.jdbc(
            url=db_url,
            table=table_name,
            properties=db_props
        ).select(*key_columns).dropDuplicates(key_columns)

        # Lấy những dòng mới chưa có trong MySQL
        new_df = df.join(
            existing_df,
            on=key_columns,
            how="left_anti"
        )

        new_count = new_df.count()

        if new_count > 0:
            new_df.write.jdbc(
                url=db_url,
                table=table_name,
                mode="append",
                properties=db_props
            )
            print(f"  --> [MySQL] Đã ghi {new_count} dòng mới vào {table_name}.")
        else:
            print(f"  --> [MySQL] Không có dòng mới cho {table_name}.")

    except Exception as e:
        print(f"  --> [MySQL] Không đọc được bảng {table_name} hoặc bảng chưa tồn tại: {e}")
        print(f"  --> [MySQL] Thử ghi toàn bộ dữ liệu batch hiện tại vào {table_name}.")

        try:
            df.write.jdbc(
                url=db_url,
                table=table_name,
                mode="append",
                properties=db_props
            )
            print(f"  --> [MySQL] Đã ghi dữ liệu vào {table_name}.")
        except Exception as write_error:
            print(f"  --> [MySQL] Lỗi ghi bảng {table_name}: {write_error}")


# ======================================================================
# HÀM GHI FACT SALES HẠN CHẾ DUPLICATE
# ======================================================================
def write_fact_sales_no_duplicate(df, table_name, key_columns):
    """
    Ghi fact_sales nhưng tránh ghi lại TransactionID đã tồn tại.
    Nếu bảng fact_sales chưa tồn tại thì ghi bình thường.
    """

    try:
        existing_df = spark.read.jdbc(
            url=db_url,
            table=table_name,
            properties=db_props
        ).select(*key_columns).dropDuplicates(key_columns)

        new_df = df.join(
            existing_df,
            on=key_columns,
            how="left_anti"
        )

        new_count = new_df.count()

        if new_count > 0:
            new_df.write.jdbc(
                url=db_url,
                table=table_name,
                mode="append",
                properties=db_props
            )
            print(f"  --> [MySQL] Đã ghi {new_count} dòng mới vào {table_name}.")
        else:
            print(f"  --> [MySQL] Không có dòng mới cho {table_name}.")

    except Exception as e:
        print(f"  --> [MySQL] Không đọc được bảng {table_name} hoặc bảng chưa tồn tại: {e}")
        print(f"  --> [MySQL] Thử ghi toàn bộ dữ liệu batch hiện tại vào {table_name}.")

        try:
            df.write.jdbc(
                url=db_url,
                table=table_name,
                mode="append",
                properties=db_props
            )
            print(f"  --> [MySQL] Đã ghi dữ liệu vào {table_name}.")
        except Exception as write_error:
            print(f"  --> [MySQL] Lỗi ghi bảng {table_name}: {write_error}")


# ======================================================================
# BƯỚC 1: LOAD VÀ CACHE DỮ LIỆU TĨNH TỪ HDFS BRONZE LAYER
# ======================================================================
try:
    WEATHER_PATH = "hdfs://namenode:9000/data/bronze/weather_optimized"
    HOLIDAYS_PATH = "hdfs://namenode:9000/data/bronze/holidays_optimized"

    print("--> Đang nạp Weather từ HDFS Bronze Layer...")
    weather_df = spark.read.parquet(WEATHER_PATH)

    print("--> Đang nạp Holidays từ HDFS Bronze Layer...")
    holidays_df = spark.read.parquet(HOLIDAYS_PATH)

    # Ép kiểu DateKey về Integer để join đúng kiểu với dữ liệu giao dịch
    weather_df = weather_df.withColumn(
        "DateKey",
        col("DateKey").cast("integer")
    )

    holidays_df = holidays_df.withColumn(
        "DateKey",
        col("DateKey").cast("integer")
    )

    # Chống duplicate trong dữ liệu bổ trợ
    weather_df = weather_df.dropDuplicates(["DateKey"])
    holidays_df = holidays_df.dropDuplicates(["DateKey"])

    # Cache lên RAM để join nhanh hơn trong streaming
    weather_df.cache()
    holidays_df.cache()

    # Kích hoạt cache
    weather_count = weather_df.count()
    holiday_count = holidays_df.count()

    print(f"--> Đã nạp và cache Weather từ HDFS: {weather_count} dòng.")
    print(f"--> Đã nạp và cache Holidays từ HDFS: {holiday_count} dòng.")

    print("--> Schema Weather:")
    weather_df.printSchema()

    print("--> Schema Holidays:")
    holidays_df.printSchema()

except Exception as e:
    print(f"--> [CẢNH BÁO] Không thể nạp dữ liệu tĩnh từ HDFS: {e}")
    raise e


# ======================================================================
# BƯỚC 2: THIẾT LẬP LUỒNG STREAMING TỪ KAFKA
# ======================================================================
schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", DoubleType(), True),
    StructField("Country", StringType(), True)
])

df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "retail_transactions") \
    .option("startingOffsets", "earliest") \
    .load()

parsed_stream = df_stream.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")


# ======================================================================
# BƯỚC 3: HÀM XỬ LÝ MICRO-BATCH
# ======================================================================
def process_lambda_architecture(batch_df, batch_id):
    print(f"\n================ BATCH ID: {batch_id} ================")

    batch_count = batch_df.count()
    print(f"--- Số dòng ban đầu: {batch_count} ---")

    if batch_count == 0:
        print("--- Batch rỗng, bỏ qua. ---")
        return

    # ---------------------------------------------------------
    # 3.1. LƯU RAW DATA XUỐNG HDFS
    # ---------------------------------------------------------
    try:
        raw_path = f"hdfs://namenode:9000/data/bronze/retail/transactions/batch_id={batch_id}"
        batch_df.write \
            .mode("overwrite") \
            .parquet(raw_path)

        print(f"  --> [HDFS] Đã lưu raw data vào Bronze Layer: {raw_path}")

    except Exception as e:
        print(f"  --> [HDFS] Lỗi ghi raw data: {e}")

    # ---------------------------------------------------------
    # 3.2. CLEAN DATA
    # ---------------------------------------------------------
    clean_df = batch_df.filter(
        col("CustomerID").isNotNull() &
        (~isnan(col("CustomerID")))
    )

    clean_df = clean_df.filter(
        (col("Quantity") > 0) &
        col("UnitPrice").isNotNull() &
        (~isnan(col("UnitPrice"))) &
        (col("UnitPrice") > 0)
    )

    clean_df = clean_df.dropDuplicates()

    clean_df = clean_df.withColumn(
        "InvoiceDate",
        to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")
    )

    clean_df = clean_df.filter(col("InvoiceDate").isNotNull())

    clean_df = clean_df.withColumn(
        "CustomerID",
        col("CustomerID").cast("integer")
    )

    clean_df = clean_df.withColumn(
        "TotalPrice",
        col("Quantity") * col("UnitPrice")
    )

    clean_df = clean_df.withColumn(
        "DateKey",
        date_format(col("InvoiceDate"), "yyyyMMdd").cast("integer")
    )

    clean_count = clean_df.count()
    print(f"--- Số dòng sau khi clean: {clean_count} ---")

    if clean_count == 0:
        print("Batch không có dữ liệu hợp lệ sau khi clean.")
        return

    # ---------------------------------------------------------
    # 3.3. JOIN STREAM DATA VỚI STATIC DATA
    # ---------------------------------------------------------
    enriched_df = clean_df.join(
        weather_df,
        on="DateKey",
        how="left"
    )

    enriched_df = enriched_df.join(
        holidays_df,
        on="DateKey",
        how="left"
    )

    enriched_df = enriched_df.fillna({
        "IsHoliday": False,
        "HolidayName": "Regular Day",
        "Temperature": 0.0,
        "Rainfall": 0.0,
        "Snowfall": 0.0
    })

    print("  --> Đã join thành công với dữ liệu Weather và Holidays.")

    # ---------------------------------------------------------
    # 3.4. TẠO STAR SCHEMA
    # ---------------------------------------------------------

    # Dimension Customer
    dim_customer = enriched_df.select(
        "CustomerID",
        "Country"
    ).dropDuplicates(["CustomerID"])

    # Dimension Product
    dim_product = enriched_df.select(
        "StockCode",
        "Description"
    ).dropDuplicates(["StockCode"])

    # Dimension Date
    # DateKey là cấp ngày nên không nên để Hour ở đây
    dim_date = enriched_df.select(
        "DateKey",
        to_date(col("InvoiceDate")).alias("FullDate"),
        year("InvoiceDate").alias("Year"),
        month("InvoiceDate").alias("Month"),
        dayofmonth("InvoiceDate").alias("Day"),
        dayofweek("InvoiceDate").alias("DayOfWeek"),
        "IsHoliday",
        "HolidayName"
    ).dropDuplicates(["DateKey"])

    # Dimension Weather
    dim_weather = enriched_df.select(
        "DateKey",
        "Temperature",
        "Rainfall",
        "Snowfall"
    ).dropDuplicates(["DateKey"])

    # Fact Sales
    # Tạo TransactionID để tránh duplicate nếu Spark chạy lại hoặc Kafka đọc lại message cũ
    fact_sales = enriched_df.withColumn(
        "TransactionID",
        sha2(
            concat_ws(
                "_",
                col("InvoiceNo"),
                col("StockCode"),
                col("CustomerID"),
                col("DateKey"),
                col("Quantity"),
                col("UnitPrice")
            ),
            256
        )
    ).select(
        "TransactionID",
        "InvoiceNo",
        "StockCode",
        "CustomerID",
        "DateKey",
        "Quantity",
        "UnitPrice",
        "TotalPrice"
    ).dropDuplicates(["TransactionID"])

    print("  --> Đã tạo Star Schema: dim_customer, dim_product, dim_date, dim_weather, fact_sales.")

    # ---------------------------------------------------------
    # 3.5. GHI XUỐNG MYSQL, CHỐNG DUPLICATE DIMENSION
    # ---------------------------------------------------------
    try:
        write_new_records_only(
            dim_customer,
            "dim_customer",
            ["CustomerID"]
        )

        write_new_records_only(
            dim_product,
            "dim_product",
            ["StockCode"]
        )

        write_new_records_only(
            dim_date,
            "dim_date",
            ["DateKey"]
        )

        write_new_records_only(
            dim_weather,
            "dim_weather",
            ["DateKey"]
        )

        write_fact_sales_no_duplicate(
            fact_sales,
            "fact_sales",
            ["TransactionID"]
        )

        print("  --> [MySQL] Hoàn tất ghi dữ liệu Star Schema cho batch hiện tại.")

    except Exception as e:
        print(f"  --> [MySQL] Lỗi tổng khi ghi Star Schema: {e}")


# ======================================================================
# BƯỚC 4: CHẠY STREAMING QUERY
# ======================================================================
query = parsed_stream.writeStream \
    .outputMode("append") \
    .foreachBatch(process_lambda_architecture) \
    .option("checkpointLocation", "hdfs://namenode:9000/checkpoints/retail_stream_no_duplicate") \
    .trigger(processingTime="10 seconds") \
    .start()

query.awaitTermination()

--> Đang nạp Weather từ HDFS Bronze Layer...
--> Đang nạp Holidays từ HDFS Bronze Layer...
--> Đã nạp và cache Weather từ HDFS: 374 dòng.
--> Đã nạp và cache Holidays từ HDFS: 10 dòng.
--> Schema Weather:
root
 |-- DateKey: integer (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Snowfall: double (nullable = true)

--> Schema Holidays:
root
 |-- DateKey: integer (nullable = true)
 |-- HolidayName: string (nullable = true)
 |-- IsHoliday: boolean (nullable = true)


================ BATCH ID: 77 ================
--- Số dòng ban đầu: 55324 ---
  --> [HDFS] Đã lưu raw data vào Bronze Layer: hdfs://namenode:9000/data/bronze/retail/transactions/batch_id=77
--- Số dòng sau khi clean: 17238 ---
  --> Đã join thành công với dữ liệu Weather và Holidays.
  --> Đã tạo Star Schema: dim_customer, dim_product, dim_date, dim_weather, fact_sales.
  --> [MySQL] Không có dòng mới cho dim_customer.
  --> [MySQL] Không có dòng mới cho dim_product.
 